# 04 · Train en_aligned and multi_aligned (DPO + LoRA)

**Runtime: L4 or A100** for the 7B model (Runtime → Change runtime type). `QUICK = True` runs a few steps on the 1.5B model and works on a T4: do that first to check the pipeline.

Both checkpoints get the **same** pairs, the same number of pairs and steps, the same LoRA config and seed (CLAUDE.md). en_aligned sees every pair in English; multi_aligned sees each pair in one language, ⅓ en, ⅓ hi, ⅓ pt.

| Step | Output on Drive (`MyDrive/mpac_hi_pt/`) | Full run, L4 |
|---|---|---|
| 1 Pairs | `training/pairs/{selected,en_aligned,multi_aligned}.jsonl` | download ~10 min, translation ~2–3 h |
| 2 Train en_aligned | `checkpoints/<model>/en_aligned/` (LoRA adapter + `training_meta.json`) | ~3–6 h |
| 3 Train multi_aligned | `checkpoints/<model>/multi_aligned/` | ~3–6 h |

Every step resumes after a disconnect: translations are cached chunk by chunk and training checkpoints every `save_steps`. Re-run cells 1–3, then the step you were on.

Pairs (D8, pilot default): PLURAL pairs stratified over countries, no country in the prompt, every eval prompt excluded. Responses ≤ 300 words, because Hindi needs about 2–3× the tokens of English.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[gpu]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
QUICK = True   # first run: 48 pairs, 6 steps, 1.5B model, T4 is fine. Set False for the full run (L4/A100).
MODEL = "qwen2.5-7b"   # full run only; must match the model you generated with in notebook 01
CONFIGS = ["configs/hi_pt.yaml"] + (["configs/hi_pt_quick.yaml"] if QUICK else [])

import torch
from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.config import get_model
from mpac.paths import resolve_root
cfg = load_config(*CONFIGS)
m = get_model(cfg, None if QUICK else MODEL)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
if "T4" in gpu:
    cfg["training"]["mt_batch_size"] = 16   # MADLAD runs in fp32 on a T4
print("model:", m["id"], "| GPU:", gpu, "| artifacts ->", resolve_root(cfg))
print({k: cfg["training"][k] for k in ("n_pairs", "budget_steps", "batch_size", "grad_accum", "max_length")})

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret set; gated models (Llama) will fail to download.")

## Step 1 · Preference pairs

Downloads PLURAL once (~2 GB), keeps the rows that fit the length limits (a few percent: most responses are ~500 words; filtering takes a few minutes), takes `n_pairs` round-robin over countries, then translates the hi / pt share with MADLAD. The selection is saved first, so a rerun never draws different pairs.

In [ ]:
from collections import Counter
from mpac.data.loaders import load_items
from mpac.training.dpo import build_preference_pairs

eval_items = load_items(load_config("configs/hi_pt.yaml"))   # the full eval set, also in QUICK mode
pairs = build_preference_pairs(cfg, items=eval_items)
for name, ps in pairs.items():
    print(name, len(ps), dict(Counter(p["lang"] for p in ps)))
print("countries:", len(Counter(p["country"] for p in pairs["en_aligned"])))

In [ ]:
# One pair per language, as the model will see it. Check the translations read sensibly.
for lang in cfg["languages"]["targets"]:
    p = next(p for p in pairs["multi_aligned"] if p["lang"] == lang)
    print(f"=========== {p['pair_id']} ({p['country']}) {lang}")
    print("PROMPT:", p["prompt"][:400], "\nCHOSEN:", p["chosen"][:400], "\nREJECTED:", p["rejected"][:400], "\n")

In [ ]:
# Token lengths with the model's chat template. `truncated` should be ~0 for every language:
# if Hindi is cut off much more than English, multi_aligned trains on partial Hindi pairs.
from transformers import AutoTokenizer
from mpac.training.dpo import length_report

tok = AutoTokenizer.from_pretrained(m["hf_name"])
def count_tokens(prompt, response):
    msgs = [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]
    text = tok.apply_chat_template(msgs, tokenize=False)   # tokenize=True returns a dict in transformers 5
    return len(tok(text, add_special_tokens=False)["input_ids"])
length_report(pairs["multi_aligned"], count_tokens, cfg["training"]["max_length"])

## Step 2 · Train en_aligned

QLoRA (4-bit base, LoRA on all linear layers); the reference model is the base with the adapter switched off. The loss is logged every 5 steps: `rewards/accuracies` should rise above 0.5 and `rewards/margins` above 0.

If the session is cut off, re-run cells 1–3 and this cell: it resumes from the last checkpoint.

In [ ]:
from mpac.training.dpo import train_dpo
print(train_dpo(cfg, m, "en_aligned", pairs["en_aligned"]))

## Step 3 · Train multi_aligned

**Runtime → Restart session first** to free the GPU, then run cells 1–3, the Step 1 cell (it loads the cached pairs in seconds) and this cell. On Colab Pro this can be a separate session or day.

In [ ]:
from mpac.training.dpo import train_dpo
print(train_dpo(cfg, m, "multi_aligned", pairs["multi_aligned"]))

In [ ]:
# Both runs must have the same budget. Compare what each one actually did.
import json
from mpac.training.dpo import adapter_dir
for ck in ("en_aligned", "multi_aligned"):
    p = adapter_dir(cfg, m["id"], ck) / "training_meta.json"
    if p.exists():
        meta = json.loads(p.read_text())
        last = [r for r in meta["log"] if "loss" in r][-1]
        print(ck, "| pairs", meta["n_pairs"], meta["langs"], "| steps", meta.get("steps", meta["log"][-1].get("step")),
              "| loss", round(last["loss"], 3), "| acc", last.get("rewards/accuracies"), "|", meta["gpu"])

## Next

Open `01_generate.ipynb`, set `CHECKPOINT = "en_aligned"` (then `"multi_aligned"`), same `QUICK` and `MODEL`, and run it: it loads the adapter and writes `generations/<model>/<checkpoint>/`. The scoring cells then compare each checkpoint's collapse with base.